In [23]:
import pandas as pd
from datetime import datetime, timedelta
from functions import *
import calendar
import warnings
warnings.filterwarnings("ignore")
import shutil

from openpyxl import load_workbook
from openpyxl.worksheet.table import Table, TableStyleInfo
from openpyxl.utils import get_column_letter

date = datetime.now().date()
# date = datetime.strptime(datetime.now(), "%d-%m-%Y").date()
base_dir_format = datetime.now().strftime('%d.%m.%y')
base_dir = f's:\\{base_dir_format}'
result_directory = 's:\\playground'
create_folder_if_not_exists(result_directory)
error_count = 0
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

ftpartialname = 'NEWCNEFINTRANSRPT'
newsalespartialname = 'CNENEWCAPTURERPT.CSV'
beindate_partial_name = 'BEINDATANEWRPT'

ftFoundFiles = search_files(base_dir,ftpartialname)
newsalesFoundFiles=search_files(base_dir,newsalespartialname)
beindata_found_files = search_files(base_dir, beindate_partial_name)

from_date = "2026-02-01"
to_date = "2026-02-09"


output_file_dealers = f'Dealers report {from_date} to {to_date}.xlsx'
output_file_ds = f'Direct Sales report {from_date} to {to_date}.xlsx'
print(ftFoundFiles[0])
print(newsalesFoundFiles[0])
print(date)

s:\09.02.26\29613905_NEWCNEFINTRANSRPT.CSV
s:\09.02.26\29614536_CNENEWCAPTURERPT.CSV
2026-02-09


In [24]:
def clean_text(s):
    words = s.split()
    if len(words[-1]) == 1 and len(words) >= 2:
        return " ".join(words[:-2])
    return " ".join(words[:-1])

def diff_month(d1, d2):
    return (d1.dt.year - d2.dt.year) * 12 + d1.dt.month - d2.dt.month

In [25]:
#load files
ft = pd.read_csv(ftFoundFiles[0],dtype='str',on_bad_lines='skip')
newsales = pd.read_csv(newsalesFoundFiles[0], dtype='str')
beindata = pd.read_csv(beindata_found_files[1],dtype="str")

# format dates 
newsales['Customer Created Date'] = pd.to_datetime(newsales['Customer Created Date'],dayfirst=True)
ft['Created Date'] = pd.to_datetime(ft['Created Date'],dayfirst=True)

In [26]:
ftt = ft.loc[ft['Created Date']==to_date]
ftt

,Subscriber Nr,Doc Type,Ftnr,Created Date,Created Time,Doc Status,Period From,Period To,Bank Date,Amount,User Name,User Fullname,Payment Ref No,Event Description,Payment Batch No,Batch Approval No,Default Entity Type,Collecting Entity,Pay Mode,Jv Type,Book Number,Smartcard,Bill Period,Bill Cycle,Invoice Type,Plan Name,Contract Number,Channel Provider,Subscriber Type,Subscriber Entity,Last Four Digits Of Card,Payment Flag
12573,18923158,Payment,3937504,2026-02-09,12:01:00 AM,Posted,NaN,NaN,NaN,1522,SAHL,SAHL,1805333,1050185517089,NaN,NaN,CNE Head Office,CNE Head office,Cash,NaN,NaN,10655749645,NaN,NaN,NaN,NaN,NaN,beIN,beIN Quartar Installment,CNE Head office,NaN,Normal payment
75238,18923158,Invoice,INV_8698454,2026-02-09,12:01:01 AM,Posted,04/02/2026 12:00:00 AM,03/05/2026 12:00:00 AM,NaN,1522,SAHL,SAHL,NaN,Bill Posted against Subscription Charge,NaN,NaN,CNE Head Office,NaN,NaN,NaN,NaN,10655749645,04/02/2026 - 03/05/2026,3M,Subscription Invoice,PREMIUM 09.24,3874678,beIN,beIN Quartar Installment,CNE Head office,NaN,NaN


In [27]:
print(str((date-timedelta(days=1))))

2026-02-08


In [30]:
ft_clone = ft.copy()
newsales_clone = newsales.copy()
beindata_clone = beindata.copy()

#                                                         filter files


ft_clone = ft_clone.loc[ft_clone['Doc Status']=='Posted']
ft_clone = ft_clone.loc[(ft_clone['Created Date'] >= from_date) & (ft_clone['Created Date'] <= to_date)]
ft_clone["Location"] = ft_clone['User Fullname'].apply(clean_text)
ft_clone["Location_Temp"] = ft_clone['Collecting Entity']


ft_clone = ft_clone.loc[ft_clone['Doc Type'].isin(['JV','Payment'])]
## link to beindata to get decoder type
merged = pd.merge(left=ft_clone,right=beindata_clone[['Decoder','Smart Card','Item Description STB']],left_on='Smartcard' ,right_on='Smart Card', how='left' )
ft_clone = merged


#### new sales
newsales_clone = newsales_clone.loc[(newsales_clone['Customer Created Date'] >= from_date) & (newsales_clone['Customer Created Date'] <= to_date)]
newsales_clone = newsales_clone.loc[~newsales_clone['Box Number'].isna()]
newsales_clone = newsales_clone.loc[~newsales_clone['Contract Status'].isna()]
newsales_clone = newsales_clone.loc[newsales_clone['Dealer Code'].str.lower().str.startswith('bein')]



# split data to (with_amount,no-amount)
newsales_clone_amount = newsales_clone.loc[~newsales_clone['Payment Amount'].isna()]
newsales_clone_no_amount = newsales_clone.loc[newsales_clone['Payment Amount'].isna()]

# link no amount with ft to get amount
merged = pd.merge(left=newsales_clone_no_amount, right=ft_clone[['Subscriber Nr','Created Date','Amount']], left_on=['Customer Number','Customer Created Date'], right_on=['Subscriber Nr','Created Date'], how='left')
merged = merged.drop(columns=['Subscriber Nr','Created Date','Payment Amount'])
merged = merged.rename(columns={'Amount':'Payment Amount'})

# concat with_amount with fixed no_amount
newsales_clone = pd.concat([merged,newsales_clone_amount])
newsales_clone= newsales_clone.drop_duplicates(keep='first')
newsales_clone['Sale Source'] = 'Dealers'
newsales_clone['Sale Type'] = 'New Sale'
newsales_clone.loc[newsales_clone['Dealer Type']=='Partner Head Office','Sale Source'] = 'Direct Sales'
newsales_clone["dataset"] = 'ft_clone'
newsales_clone['Location'] =newsales_clone['Dealer Name']
newsales_clone['Location_Temp'] = ""
newsales_clone['Staff #'] = ""




# link with beindata to get SC,box type
merged = pd.merge(left=newsales_clone,right=beindata_clone[['Decoder','Smart Card','Item Description STB']], left_on='Box Number',right_on='Decoder',how='left')
merged.drop(columns=['Decoder'])
newsales_clone = merged

# link with FT to get MOP, FTNR, Bill Period

merged = pd.merge(left=newsales_clone,right=ft_clone[['Subscriber Nr','Created Date','Amount','Pay Mode','Ftnr','Bill Period']], left_on=['Customer Number','Customer Created Date','Payment Amount'], right_on=['Subscriber Nr','Created Date','Amount'],how='left')
merged.drop(columns=['Subscriber Nr','Created Date','Amount'])
merged.loc[merged['Pay Mode'].isna(),'Pay Mode'] = 'Cash'
newsales_clone = merged

newsales_clone['Start Date'] = pd.to_datetime(newsales_clone['Start Date'],dayfirst=True)
newsales_clone['End Date'] = pd.to_datetime(newsales_clone['End Date'],dayfirst=True)

newsales_clone['Contract Period'] = '12M' #diff_month(newsales_clone['End Date'] , newsales_clone['Start Date'])




#### FT

# split into dealers , direct sales

ft_clone_dealers = ft_clone.loc[ft_clone['Default Entity Type'].isin(['Bein Dealer'])]
ft_clone_dealers = ft_clone_dealers.loc[ft_clone_dealers['Doc Type']=='JV']
ft_clone_dealers['Sale Source'] = 'Dealers'
ft_clone_dealers['Sale Type'] = 'Transaction'
ft_clone_dealers['dataset'] = 'ft_clone_dealers'

ft_clone_ds = ft_clone.loc[ft_clone['Default Entity Type'].isin(['Partner Head Office'])]
ft_clone_ds = ft_clone_ds.loc[ft_clone_ds['Doc Type']=='Payment']
ft_clone_ds['Sale Source'] = 'Direct Sales'
ft_clone_ds['Sale Type'] = 'Transaction'
ft_clone_ds['dataset'] = 'ft_clone_ds'




# ft_clone = ft_clone.loc[ft_clone['Default Entity Type'].isin(['Bein Dealer','Partner Head Office'])]



#Format the final results
# Transaction Date	LOCATION	AGENT	SERIAL	PACKAGE	PAYMENT PLAN	MOP	SALETYPE	BOX TYPE	TRANSACTION DATE	CSD	CED	AMOUNT	STAFF#	INVOICE	CONTRACT PERIOD	SALE TYPE
cols =  ['TRANSACTION DATE','LOCATION','AGENT','SERIAL','PAYMENT PLAN','MOP','BOX TYPE','CSD', 'CED','AMOUNT', 'STAFF #', 'FTNR','CONTRACT PERIOD','SALE SOURCE','SALE TYPE','Location_Temp']



newsales_clone = newsales_clone[['Customer Created Date','Location','User Name','Smart Card','Plan','Pay Mode','Item Description STB','Start Date', 'End Date','Payment Amount','Staff #', 'Ftnr','Contract Period','Sale Source','Sale Type','Location_Temp']]
newsales_clone['Contract Period'] = '12M' #diff_month(newsales_clone['End Date'] , newsales_clone['Start Date'])
newsales_clone.columns=cols
newsales_clone.to_csv('newsales.csv',index=False)

ft_final = pd.concat([ft_clone_dealers,ft_clone_ds])
ft_final['Start Date'] = ''
ft_final['End Date'] = ''
ft_final['Contract Period'] = ''
ft_final['Staff #'] = ''
ft_final['Contract Period'] = '12M'




ft_final = ft_final[['Created Date','Location','User Name', 'Smartcard', 'Plan Name','Pay Mode','Item Description STB','Start Date', 'End Date', 'Amount','Staff #','Ftnr','Contract Period','Sale Source','Sale Type','Location_Temp']]
ft_final = ft_final.drop_duplicates()
ft_final.loc[ft_final['Pay Mode'].isna(),'Pay Mode'] = 'Cash'
ft_final.columns = cols



ft_final.to_csv('ft_final.csv',index=False)

all_final = pd.concat([ft_final,newsales_clone])
all_final = all_final.reset_index(drop=True)
all_final = all_final.drop_duplicates()
all_final['TRANSACTION DATE'] = pd.to_datetime(all_final['TRANSACTION DATE'],dayfirst=True)


all_final.loc[(all_final['SALE TYPE']=='Transaction') & (all_final['SALE SOURCE']=='Direct Sales'),'LOCATION']=all_final['Location_Temp']

all_final = all_final.drop(columns=['Location_Temp'])
all_final['PACKAGE'] = ''
sorted_cols = ['TRANSACTION DATE', 'LOCATION', 'AGENT', 'SERIAL','PACKAGE', 'PAYMENT PLAN',
       'MOP', 'SALE TYPE', 'BOX TYPE', 'CSD', 'CED', 'AMOUNT', 'STAFF #', 'FTNR',
       'CONTRACT PERIOD', 'SALE SOURCE']

all_final = all_final[sorted_cols]
all_final = all_final.sort_values('TRANSACTION DATE')

all_final_dealers = all_final[all_final['SALE SOURCE']=='Dealers']
all_final_ds = all_final[all_final['SALE SOURCE']=='Direct Sales']

all_final_dealers = all_final_dealers.drop("SALE SOURCE",axis = 1)
all_final_ds = all_final_ds.drop("SALE SOURCE",axis = 1)





# -------------------------------------------------
# FUNCTION: Format Excel (table + autofit)
# -------------------------------------------------
def format_excel_table(file_path, table_name):

    wb = load_workbook(file_path)
    ws = wb["Data"]

    end_row = ws.max_row
    end_col = ws.max_column

    table = Table(
        displayName=table_name,
        ref=f"A1:{get_column_letter(end_col)}{end_row}"
    )

    style = TableStyleInfo(
        name="TableStyleMedium9",
        showFirstColumn=False,
        showLastColumn=False,
        showRowStripes=True,
        showColumnStripes=False
    )

    table.tableStyleInfo = style
    ws.add_table(table)

    # Auto-fit columns
    for col_idx in range(1, end_col + 1):
        col_letter = get_column_letter(col_idx)
        max_length = 0

        for cell in ws[col_letter]:
            if cell.value is not None:
                max_length = max(max_length, len(str(cell.value)))

        ws.column_dimensions[col_letter].width = max_length + 5

    wb.save(file_path)


# -------------------------------------------------
# EXPORT DEALERS
# -------------------------------------------------

with pd.ExcelWriter(output_file_dealers, engine="openpyxl") as writer:
    all_final_dealers.to_excel(writer, sheet_name="Data", index=False)

format_excel_table(output_file_dealers, "DataTableDealers")


# -------------------------------------------------
# EXPORT DIRECT SALES
# -------------------------------------------------

with pd.ExcelWriter(output_file_ds, engine="openpyxl") as writer:
    all_final_ds.to_excel(writer, sheet_name="Data", index=False)

format_excel_table(output_file_ds, "DataTableDS")



In [ ]:
all_final.columns

In [ ]:
newsales.loc[newsales['Customer Number']=='19232125']



In [ ]:
ft_clone.loc[ft_clone['Subscriber Nr']=='19232125']


In [ ]:
newsales_clone